## Setup

In [75]:
from googleapiclient.discovery import build
import json
import re
from collections import Counter
from datetime import datetime
from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import glob
import os

In [76]:
with open ("../performative.txt", "r") as f:
    lines = f.readlines()
    API_KEY = lines[2].strip()
    
yt = build("youtube", "v3", developerKey=API_KEY)

### Static/Maps

In [77]:
GENRE_MAP = {
    "1":  "Film & Animation",
    "2":  "Autos & Vehicles",
    "10": "Music",
    "15": "Pets & Animals",
    "17": "Sports",
    "19": "Travel & Events",
    "20": "Gaming",
    "22": "People & Blogs",
    "23": "Comedy",
    "24": "Entertainment",
    "25": "News & Politics",
    "26": "Howto & Style",
    "27": "Education",
    "28": "Science & Technology",
    "29": "Nonprofits & Activism",
}

In [78]:
QUERIES = [
    "video essay film analysis",
    "philosophy explained documentary",
    "internet culture deep dive",
    "history documentary essay channel",
    "game design analysis",
]
MAX_PER_QUERY = 50  # how many channels to collect per keyword/phrase


## Scripts

In [79]:
def search_channels_queries(query, max_results, seen):
    collected = []
    next_page = None

    while len(collected) < max_results:
        resp = yt.search().list(
            part="snippet",
            q = query, 
            type="channel",
            relevanceLanguage = "en",
            maxResults=50,
            pageToken = next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"]["channelId"]
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   item["snippet"]["channelId"],
                    "channel_name": item["snippet"]["title"],
                })
            if len(collected) >= max_results:
                break
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

In [80]:
def search_channels_playlist(playlist_id, seen):
    collected = []
    next_page = None

    while True:
        resp = yt.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page,
        ).execute()

        for item in resp["items"]:
            cid = item["snippet"].get("videoOwnerChannelId")  # dict[key] raises a KeyError when the key is missing. dict.get(key) gracefully returns None. verify?
            if cid not in seen:
                seen.add(cid)
                collected.append({
                    "channel_id":   cid,
                    "channel_name": item["snippet"]["videoOwnerChannelTitle"],
                })
        
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return collected

## Running

In [81]:
all_channels = []
seen = set()

for query in QUERIES:
    print(f"Searching: {query}")
    results = search_channels_queries(query, MAX_PER_QUERY, seen)
    print(f" got {len(results)} channels")
    all_channels.extend(results)

with open("../data/processed/ve_channels/ve_channels_queries.json", "r") as f:
    existing = json.load(f)
combined = existing + all_channels
with open("../data/processed/ve_channels_queries.json", "w") as f:
    json.dump(all_channels, f, indent=2)

Searching: video essay film analysis
 got 50 channels
Searching: philosophy explained documentary
 got 50 channels
Searching: internet culture deep dive
 got 50 channels
Searching: history documentary essay channel
 got 50 channels
Searching: game design analysis
 got 50 channels


In [82]:
seen = set()

results = search_channels_playlist("PLXz6Pf8rae1mIGQ8IdpggWP2RmW1wmXsa", seen)

with open("../data/processed/ve_channels/ve_channels_playlist.json", "w") as f:
    json.dump(results, f, indent=2)

KeyError: 'videoOwnerChannelTitle'

In [ ]:
results2 = search_channels_playlist("PLbOF9TQmQrNgWHaLCPaE81l4hEL-2Rrjf", set())

with open("../data/processed/ve_channels/ve_channels_playlist_2.json", "w") as f:
    json.dump(results2, f, indent=2)

## Viewing Data

In [83]:
with open("../data/processed/ve_channels/ve_channels_queries.json", "r") as f:
    data_queries = json.load(f)

In [84]:
df_queries = pd.DataFrame(data_queries)
df_queries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    225 non-null    object
 1   channel_name  225 non-null    object
dtypes: object(2)
memory usage: 3.6+ KB


In [85]:
with open("../data/processed/ve_channels/ve_channels_playlist.json", "r") as f:
    data_playlist = json.load(f)

In [86]:
df_playlist = pd.DataFrame(data_playlist)
df_playlist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    106 non-null    object
 1   channel_name  106 non-null    object
dtypes: object(2)
memory usage: 1.8+ KB


In [87]:
with open("../data/processed/ve_channels/ve_channels_playlist_2.json", "r") as f:
    data_playlist2 = json.load(f)

In [88]:
df_playlist2 = pd.DataFrame(data_playlist2)
df_playlist2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    23 non-null     object
 1   channel_name  23 non-null     object
dtypes: object(2)
memory usage: 500.0+ bytes


## Cleaning Data

In [89]:
df = pd.concat([df_queries, df_playlist, df_playlist2])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 354 entries, 0 to 22
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    354 non-null    object
 1   channel_name  354 non-null    object
dtypes: object(2)
memory usage: 8.3+ KB


In [90]:
df = df.drop_duplicates(subset="channel_id").reset_index(drop=True)

In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 340 entries, 0 to 339
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    340 non-null    object
 1   channel_name  340 non-null    object
dtypes: object(2)
memory usage: 5.4+ KB


In [97]:
rows_to_add = [{"channel_id": "UCD6TPU-PvTMvqgzC_AM7_uA", "channel_name": "The People Profiles"}]

In [ ]:
rows_to_drop = [11, 12, 18, 36, 50, 51, 52, 54]

In [98]:
df[df["channel_id"] == "UC1DTYW241WD64ah5BFWn4JA"]

,channel_id,channel_name


In [95]:
df.iloc[50:100]

,channel_id,channel_name
50,UCwbDCAPugfe1KrbCZ9F0VPw,Philosophy Origins
51,UCTNKkxZDNDjVGcibLg1D6Ww,Philosophy Matters
52,UCCfI-j18jSgT45svMcofILw,Documentaries and Philosophy
53,UCp1mRTkVlqDnxz_9S0YD9YQ,Philosophies for Life
54,UCIuCDa-yyDddjzVfu6WMiNQ,Philosophy Documentaries
55,UCBz-uxcltpYVrL9On42z2qQ,Philosophy Corner
56,UCVvux6AY30Iqe7bxxcjL2LA,Sam's Philosophy
57,UCXRNtFb3LCHxaV4u0XlrAvQ,Philosophy Sleeping Story
58,UC7IcJI8PUf5Z3zKxnZvTBog,The School of Life
59,UCWTc-B9-9j5Tykq7ou4tL3Q,Philosophy Decoded
